# Step-Video-T2V gesture clips (run OFFLINE on a Colab GPU)
**Status: template - NOT executed by the author.** Step-Video-T2V (30B parameters) needs a large GPU (the model card / Neurohive article recommend ~80 GB VRAM; 4-8 GPUs or quantisation for smaller setups), so it could not run in the sandbox where the rest of this repo was built. Follow the official repo README for the exact install / run commands: https://github.com/stepfun-ai/Step-Video-T2V (weights: https://huggingface.co/stepfun-ai/stepvideo-t2v, Turbo: .../stepvideo-t2v-turbo).

Output of this notebook: `clips/<class>/*.mp4`. Then locally:
```
python src/ingest_clips.py clips data/t2v_frames --frames 6
```
and add `data/t2v_frames` to the TRAINING set only (see `src/experiments.py::train_set`). **Label every generated clip as SYNTHETIC; do not generate identifiable real people.**

In [ ]:
# 1) Setup - follow the repo README (versions change). Typical shape:
# !git clone https://github.com/stepfun-ai/Step-Video-T2V && cd Step-Video-T2V && pip install -e .
# !huggingface-cli download stepfun-ai/stepvideo-t2v-turbo --local-dir ./weights   # Turbo: 10-15 steps, CFG ~5


In [ ]:
# 2) Prompt grid: gesture x environment x camera. Hands only (no faces) -> avoids identity/consent issues.
import itertools, json
GESTURES = {
 'peace': 'a close-up of a bare hand making a peace sign, index and middle finger spread in a V, other fingers folded',
 'thumb': 'a close-up of a bare hand giving a thumbs-up, thumb extended, other fingers closed in a fist',
 'stop' : 'a close-up of an open bare palm facing the camera, all five fingers spread, like a stop sign',
 'ok'   : 'a close-up of a bare hand making the OK sign, thumb and index finger forming a circle',
}
ENVS = ['on a grey office table, soft daylight', 'on dark tiled floor, dim lighting', 'on a street granite pavement, overcast',
        'against a bright window, backlit', 'on a white floor, strong shadows', 'in a car, moving light']
CAMS = ['handheld camera, slight shake', 'slow zoom in', 'slow pan left to right', 'fast camera move with motion blur']
prompts = [dict(cls=c, env=e, cam=m, text=f'{t}, {e}, {m}, photorealistic, 4k') for (c,t),e,m in itertools.product(GESTURES.items(), ENVS, CAMS)]
print(len(prompts), 'prompts'); json.dump(prompts, open('prompts.json','w'), indent=1)


In [ ]:
# 3) Generate (pseudo-code: replace generate_video(...) with the repo's inference call; keep clips SHORT, e.g. 33-65 frames, 544x992 or lower)
import os
for i,p in enumerate(prompts):
    out = f"clips/{p['cls']}/{i:03d}.mp4"; os.makedirs(os.path.dirname(out), exist_ok=True)
    # generate_video(prompt=p['text'], out_path=out, steps=15, cfg_scale=5.0)   # <- Turbo settings from the model card


In [ ]:
# 4) Zip and download -> unzip next to the repo, then run src/ingest_clips.py
# !zip -r clips.zip clips && # from google.colab import files; files.download('clips.zip')


## Optional: MobileNetV2 transfer learning (A1 section 5)
Run the A1 notebook cell with ImageNet weights on the same folds; compare with the table in the report. Not run in this repo (no TensorFlow / internet in the build sandbox).